# 05 — RAG: Retrieval-Augmented Generation

**Tickets:** R-01, R-02, R-03, R-04, R-05, R-06  
**Business Question (BQ-5):** Can a conversational interface answer ad-hoc questions about NYC taxi operations & policy?  
**Purpose:** Prepare a document corpus, generate embeddings, build a retrieval function, and wire up an LLM to answer natural-language questions.

---

## Setup

In [0]:
# TODO: initialise SparkSession / Databricks context
# TODO: install / import embedding model and vector store libraries

## R-01 & R-02 — Corpus preparation & chunking

In [0]:
# R-01 — Collect & prepare RAG corpus documents
corpus_documents = [
    {
        "doc_id": "tlc_rules",
        "source": "NYC Taxi & Limousine Commission",
        "title": "TLC Rules & Regulations",
        "content": (
            "All NYC yellow taxis must operate with a valid TLC medallion. "
            "Drivers must hold a valid TLC driver license. "
            "The taximeter must be used for all trips and must be engaged at the start of each ride. "
            "Drivers must accept any passenger requesting a trip to any destination within the five boroughs of New York City. "
            "The passenger has the right to choose the method of payment (cash or card). "
            "Air conditioning is required from May 1 through October 31 when the outside temperature exceeds 70°F. "
            "Drivers must offer a printed receipt at the end of every trip. "
            "Drivers may not refuse passengers based on their destination."
        ),
    },
    {
        "doc_id": "taxi_faqs",
        "source": "NYC Taxi & Limousine Commission",
        "title": "NYC Taxi Frequently Asked Questions",
        "content": (
            "Yellow cabs can be street-hailed in all five boroughs of New York City. "
            "Green Boro Taxis serve the outer boroughs and upper Manhattan above 110th Street. "
            "Accepted payment methods include cash, credit or debit card, and contactless or mobile payments. "
            "Tipping is not required but is customary at 15–20% of the fare. "
            "For lost property in a taxi, call 311. "
            "To file a complaint about a driver or trip, contact the TLC by calling 311 or visiting nyc.gov/tlc. "
            "Wheelchair-accessible taxis are available across the city."
        ),
    },
    {
        "doc_id": "pricing_policy",
        "source": "NYC Taxi & Limousine Commission",
        "title": "NYC Taxi Pricing Policy",
        "content": (
            "The initial metered fare charge is $3.00. "
            "The meter rate is $0.70 per 1/5 mile traveled. "
            "The meter also charges $0.70 per 60 seconds when the taxi is in slow traffic or idle. "
            "A rush hour surcharge of $2.50 applies Monday through Friday from 4:00 PM to 8:00 PM. "
            "An overnight surcharge of $1.00 applies from 8:00 PM to 6:00 AM. "
            "An MTA State Surcharge of $0.50 is added to every ride. "
            "An improvement surcharge of $1.00 is added to every ride. "
            "A congestion surcharge of $2.50 applies to all trips that begin, end, or pass through Manhattan below 96th Street. "
            "The flat fare from JFK Airport to Manhattan is $70.00, plus tolls and tip. "
            "Trips to Newark Airport are charged at the metered fare plus a $20.00 surcharge. "
            "All tolls incurred during the trip are the responsibility of the passenger."
        ),
    },
]

print(f"✓ R-01 complete: {len(corpus_documents)} corpus documents prepared")
for doc in corpus_documents:
    print(f"  - {doc['doc_id']}: {doc['title']} ({len(doc['content'])} chars)")

In [0]:
# R-02 — Chunk documents into retrieval-friendly segments

def chunk_document(doc, max_chars=1600, overlap_chars=200):
    """Split a document into chunks on sentence boundaries with overlap.
    
    Target: 300-500 tokens (~1200-2000 chars at ~4 chars/token).
    Default max_chars=1600 (~400 tokens), overlap=200 (~50 tokens).
    """
    text = doc["content"]
    sentences = [s.strip() + "." for s in text.split(".") if s.strip()]
    
    chunks = []
    current_chunk = ""
    chunk_idx = 0

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append({
                "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
                "doc_id": doc["doc_id"],
                "source": doc["source"],
                "title": doc["title"],
                "content": current_chunk.strip(),
            })
            # Keep overlap from end of current chunk
            overlap_text = current_chunk[-overlap_chars:] if len(current_chunk) > overlap_chars else current_chunk
            current_chunk = overlap_text + " " + sentence
            chunk_idx += 1
        else:
            current_chunk = (current_chunk + " " + sentence).strip()

    if current_chunk.strip():
        chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{chunk_idx}",
            "doc_id": doc["doc_id"],
            "source": doc["source"],
            "title": doc["title"],
            "content": current_chunk.strip(),
        })

    return chunks


# Chunk all corpus documents
corpus_chunks = []
for doc in corpus_documents:
    chunks = chunk_document(doc)
    corpus_chunks.extend(chunks)

print(f"✓ R-02 complete: {len(corpus_chunks)} chunks from {len(corpus_documents)} documents")
for chunk in corpus_chunks:
    print(f"  - {chunk['chunk_id']}: {len(chunk['content'])} chars (~{len(chunk['content']) // 4} tokens)")

## R-03 — Generate embeddings & store as vector table

In [0]:
# TODO: generate embeddings for each chunk
# TODO: store as a Delta / vector search table

## R-04 — Retrieval function

In [0]:
# TODO: implement retrieve(query, top_k=5) — embed query, return top-k chunks

## R-05 — RAG pipeline (prompt + retrieval + LLM)

In [0]:
# TODO: build prompt template that injects retrieved context
# TODO: call LLM with prompt; return answer

## R-06 — Test with sample questions

In [0]:
sample_questions = [
    "What is the standard taxi rate in NYC?",
    "When is taxi demand highest in Manhattan?",
    "How is the MTA tax applied to taxi fares?",
    "What payment types do NYC taxis accept?",
    "What is the JFK flat rate?",
]

# TODO: run each question through the RAG pipeline and print answers
# TODO: document quality observations for each answer

## Answer quality log

| Question | Answer quality | Notes |
|----------|---------------|-------|
| | | |